In [82]:
!pip3 install openpyxl pandas langchain langchain-openai langchain-ollama python-dotenv

from context_processing.col_row_context import col_based_processing, row_based_processing
from context_processing.raw_dump import get_read_data
from context_processing.inverted_index import build_inverted_index
from typing import Union


import openpyxl
import json
import pandas as pd
from llm_util import call_llm
def get_data(path="all_data_912/dataset.json"):
    df=pd.read_json(path)
    return df
# given a context it should give a respone
def eval(context):
    prompt=f"Given the context {context}, answer the question"
    response=call_llm(prompt)
    return response
def list_sheets(path: str):
    wb = openpyxl.load_workbook(path, data_only=True)
    return wb.sheetnames
def make_context(spreadsheet_path,sheet_name:Union[int,str,None]=None,type="col"):
    try:
    
        if sheet_name is None:
            sheets=list_sheets(spreadsheet_path)
        else :
            sheets=[sheet_name]
        context={}
        for sheet in sheets:
            if type=="col":
                context[sheet]= col_based_processing(spreadsheet_path,sheet)
            elif type=="row":
                context[sheet]= row_based_processing(spreadsheet_path,sheet)
            elif type=="inverted":
                context[sheet]= build_inverted_index(spreadsheet_path,sheet)
            elif type=="raw":
                context[sheet]= get_read_data(spreadsheet_path,sheet)
    except Exception as e:
        return None
    return context
# different benchmarks for col/row/inverted/raw
# shuld have a feature for single llm or multiple llms evaluation


def eval_benchmark():
    df=get_data()
    df['context']=df['spreadsheet_path'].apply(make_context)
    df['eval_response']=df['context'].apply(eval)
    df['answer_correctness']= df.apply(lambda row: row['eval_response'].strip().lower() == row['answer'].strip().lower(), axis=1)
    return df 


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [83]:
print(make_context("test.xlsx",type="row"))
with open("eval_output.json", "w") as f:
    json.dump(make_context("test.xlsx"), f, indent=2)

{'Export Summary': '|A1, None, empty| |B1, None, empty| |C1, None, empty| |D1, None, empty| || |A2, None, empty| |B2, None, empty| |C2, None, empty| |D2, None, empty| || |A3, None, empty| |B3, This document was exported from Numbers.  Each table was converted to an Excel worksheet. All other objects on each Numbers sheet were placed on separate worksheets. Please be aware that formula calculations may differ in Excel., string| |C3, None, empty| |D3, None, empty| || |A4, None, empty| |B4, None, empty| |C4, None, empty| |D4, None, empty| || |A5, None, empty| |B5, None, empty| |C5, None, empty| |D5, None, empty| || |A6, None, empty| |B6, None, empty| |C6, None, empty| |D6, None, empty| || |A7, None, empty| |B7, Numbers Sheet Name, string| |C7, Numbers Table Name, string| |D7, Excel Worksheet Name, string| || |A8, None, empty| |B8, None, empty| |C8, None, empty| |D8, None, empty| || |A9, None, empty| |B9, Salary List, string| |C9, None, empty| |D9, None, empty| || |A10, None, empty| |B10, 

In [84]:
df=get_data('evaluation/all_data_912/dataset.json')


In [85]:
print(len(df))

912


In [86]:
import os
def give_sheets_from_dir(dir_path):
    result = {}
    for filename in os.listdir(dir_path):
        if filename.endswith("input.xlsx") or filename.endswith(".xlsm") or filename.endswith(".xltx") or filename.endswith(".xltm"):
            file_path = os.path.join(dir_path, filename)
            identity = filename.strip('_input.xlsx')
            result[file_path] = identity
    return result

print(give_sheets_from_dir("evaluation/all_data_912/spreadsheet/13-1"))

{'evaluation/all_data_912/spreadsheet/13-1/1_13-1_input.xlsx': '1_13-1', 'evaluation/all_data_912/spreadsheet/13-1/3_13-1_input.xlsx': '3_13-1', 'evaluation/all_data_912/spreadsheet/13-1/2_13-1_input.xlsx': '2_13-1'}


In [87]:
import pandas as pd

def expand_spreadsheets(df):
    rows = []
    for _, row in df.iterrows():
        # get dict of {path: identity}
        mapping = give_sheets_from_dir(f"evaluation/all_data_912/{row['spreadsheet_path']}")
        
        # expand into multiple rows
        for path, identity in mapping.items():
            new_row = row.to_dict().copy()
            new_row['new_spreadsheet_path'] = path
            new_row['new_id'] = identity
            rows.append(new_row)
    
    return pd.DataFrame(rows)

# Usage:
df_expanded = expand_spreadsheets(df)
df_expanded.sort_values(by='new_id', inplace=True)
df=df_expanded
df

,id,instruction,spreadsheet_path,instruction_type,answer_position,answer_sheet,data_position,new_spreadsheet_path,new_id
285,102-20,I need guidance with reformatting my Excel raw...,spreadsheet/102-20,Sheet-Level Manipulation,'Raw Data'!B2:I40,Raw Data,"Raw Data!'A1:J19,'Expected result!'A1:J32,",evaluation/all_data_912/spreadsheet/102-20/1_1...,1_102-20
1653,10281,I want to configure column 'J' under 'Densitie...,spreadsheet/10281,Cell-Level Manipulation,J15:J17,NaN,NaN,evaluation/all_data_912/spreadsheet/10281/1_10...,1_10281
1656,10452,I need a method to perform a vertical lookup i...,spreadsheet/10452,Cell-Level Manipulation,E4:E12,NaN,NaN,evaluation/all_data_912/spreadsheet/10452/1_10...,1_10452
1657,10493,I am working on a salary spreadsheet for my wi...,spreadsheet/10493,Cell-Level Manipulation,C8:G11,NaN,NaN,evaluation/all_data_912/spreadsheet/10493/1_10...,1_10493
288,105-13,How can I use VBA to filter data from a multi-...,spreadsheet/105-13,Sheet-Level Manipulation,'Calculation!'C11:I490',"Source,Calculation","Source!'H3:K1964,'Calculation!'B11:R490'",evaluation/all_data_912/spreadsheet/105-13/1_1...,1_105-13
...,...,...,...,...,...,...,...,...,...
958,CF_7938,How can I use conditional formatting to highli...,spreadsheet/CF_7938,Sheet-Level Manipulation,'Sheet1'!C8:I8,Sheet1,B4:J8,evaluation/all_data_912/spreadsheet/CF_7938/3_...,3_CF_7938
963,CF_8276,How can I set up a conditional format to color...,spreadsheet/CF_8276,Sheet-Level Manipulation,'Data'!E2:E8,Data,A1:I4,evaluation/all_data_912/spreadsheet/CF_8276/3_...,3_CF_8276
966,CF_8830,I am needing to highlight cells in red where f...,spreadsheet/CF_8830,Sheet-Level Manipulation,'Sheet1'!B2:B8,Sheet1,A1:A8,evaluation/all_data_912/spreadsheet/CF_8830/3_...,3_CF_8830
969,CF_9945,I need to apply conditional formatting to cell...,spreadsheet/CF_9945,Sheet-Level Manipulation,'Sheet1'!A3:G3,Sheet1,A1:G13,evaluation/all_data_912/spreadsheet/CF_9945/3_...,3_CF_9945


In [88]:
print(len(df))

2726


In [89]:
df['context']=df['new_spreadsheet_path'].apply(make_context)
# df['eval_response']=df['context'].apply(eval)

/Users/karman/.pyenv/versions/3.10.0/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/karman/.pyenv/versions/3.10.0/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/Users/karman/.pyenv/versions/3.10.0/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/Users/karman/.pyenv/versions/3.10.0/lib/python3.10/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/karman/.pyenv/versions/3.10.0/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Slicer List extension is not supported and will be removed
  warn(msg)
/Users/karma

In [90]:
df

,id,instruction,spreadsheet_path,instruction_type,answer_position,answer_sheet,data_position,new_spreadsheet_path,new_id,context
285,102-20,I need guidance with reformatting my Excel raw...,spreadsheet/102-20,Sheet-Level Manipulation,'Raw Data'!B2:I40,Raw Data,"Raw Data!'A1:J19,'Expected result!'A1:J32,",evaluation/all_data_912/spreadsheet/102-20/1_1...,1_102-20,"{'Raw Data': '|A1, FX Fund, string| |A2, SOI, ..."
1653,10281,I want to configure column 'J' under 'Densitie...,spreadsheet/10281,Cell-Level Manipulation,J15:J17,NaN,NaN,evaluation/all_data_912/spreadsheet/10281/1_10...,1_10281,"{'Sheet1': '|A1, Proctors, string| |A2, Procto..."
1656,10452,I need a method to perform a vertical lookup i...,spreadsheet/10452,Cell-Level Manipulation,E4:E12,NaN,NaN,evaluation/all_data_912/spreadsheet/10452/1_10...,1_10452,"{'Sheet1': '|A1, None, empty| |A2, None, empty..."
1657,10493,I am working on a salary spreadsheet for my wi...,spreadsheet/10493,Cell-Level Manipulation,C8:G11,NaN,NaN,evaluation/all_data_912/spreadsheet/10493/1_10...,1_10493,"{'Sheet1': '|A1, Employee, string| |A2, Jo, st..."
288,105-13,How can I use VBA to filter data from a multi-...,spreadsheet/105-13,Sheet-Level Manipulation,'Calculation!'C11:I490',"Source,Calculation","Source!'H3:K1964,'Calculation!'B11:R490'",evaluation/all_data_912/spreadsheet/105-13/1_1...,1_105-13,"{'Source': '|A1, None, empty| |A2, None, empty..."
...,...,...,...,...,...,...,...,...,...,...
958,CF_7938,How can I use conditional formatting to highli...,spreadsheet/CF_7938,Sheet-Level Manipulation,'Sheet1'!C8:I8,Sheet1,B4:J8,evaluation/all_data_912/spreadsheet/CF_7938/3_...,3_CF_7938,"{'Sheet1': '|A1, None, empty| |A2, None, empty..."
963,CF_8276,How can I set up a conditional format to color...,spreadsheet/CF_8276,Sheet-Level Manipulation,'Data'!E2:E8,Data,A1:I4,evaluation/all_data_912/spreadsheet/CF_8276/3_...,3_CF_8276,"{'Data': '|A1, None, empty| |A2, Fred, string|..."
966,CF_8830,I am needing to highlight cells in red where f...,spreadsheet/CF_8830,Sheet-Level Manipulation,'Sheet1'!B2:B8,Sheet1,A1:A8,evaluation/all_data_912/spreadsheet/CF_8830/3_...,3_CF_8830,"{'Sheet1': '|A1, Vehicle Description, string| ..."
969,CF_9945,I need to apply conditional formatting to cell...,spreadsheet/CF_9945,Sheet-Level Manipulation,'Sheet1'!A3:G3,Sheet1,A1:G13,evaluation/all_data_912/spreadsheet/CF_9945/3_...,3_CF_9945,"{'Sheet1': '|A1, Veke 1, string| |A2, 2022-01-..."


In [91]:
df.to_csv("staging.csv", index=False)